# 🎬 Taller GenMedia con Gemini: De una Idea a una Producción Multimedia

En este taller construiremos una producción audiovisual coherente y completa: **Brief JSON estructurado, Keyframe 16:9 en alta definición, Locución dirigida con Voice Director, Banda sonora original y Clips cinemáticos con animación y reemplazo de personajes**.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alarcon7a/gemini_media/blob/main/codelab-genmedia-001/tutorial_genmedia_gemini_omni.ipynb)
[![Open in Colab Enterprise](https://img.shields.io/badge/Open%20in-Colab%20Enterprise-1a73e8?style=flat&logo=googlecloud&logoColor=white)](https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Falarcon7a%2Fgemini_media%2Fmain%2Fcodelab-genmedia-001%2Ftutorial_genmedia_gemini_omni.ipynb)

> **Resultado**: Directorio `media/taller_genmedia/` con `brief.json`, `01_keyframe.png`, `02_locucion.wav`, `03_musica.mp3`, `04_omni_texto_video.mp4`, `05_omni_keyframe_video.mp4` y `06_omni_character_swap.mp4`.


## 🗺️ Agenda & Single Source of Truth (SSoT)

| Minutos | Bloque | Modelo Oficial | Entregable |
|---:|---|---|---|
| 0–8 | **Setup & Conexión** | SDK `google-genai` | Cliente autenticado (Enterprise o API Key) |
| 8–18 | **Brief Creativo Maestro** | `gemini-3.7-flash` | `brief.json` estructurado con Pydantic |
| 18–30 | **Keyframe Visual 16:9** | `gemini-3.1-flash-image` (Nano Banana 2) | `01_keyframe.png` en 5 capas |
| 30–38 | **Voice Director TTS** | `gemini-3.1-flash-tts-preview` | `02_locucion.wav` con emotion tags |
| 38–46 | **Banda Sonora Original** | `lyria-3-clip-preview` | `03_musica.mp3` en 72 BPM |
| 46–58 | **Video Cinemático & Swap**| `gemini-omni-flash-preview` | Videos MP4 (First-Frame + Character Swap) |
| 58–60 | **Checklist & Reto** | Validación | Inventario completo de assets |

Una sola fuente de verdad: el brief JSON maestro orquesta y sincroniza todas las modalidades.


In [ ]:
# Instalación del SDK oficial google-genai y utilidades multimedia
%pip install -Uq "google-genai>=2.20.0" pydantic pillow google-auth


In [ ]:
import base64
import json
import mimetypes
import os
import subprocess
import urllib.request
import wave
from pathlib import Path

import google.auth
from google.auth.transport.requests import Request as GoogleAuthRequest
from google import genai
from pydantic import BaseModel, Field
from IPython.display import Audio, Image as DisplayImage, Video, HTML, display

# =========================================================================
# 🔐 CONFIGURACIÓN DE CLIENTE (HÍBRIDO: Colab Enterprise + Google AI Studio)
# =========================================================================
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

if not PROJECT_ID and not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY", "")
    except Exception:
        pass

if not PROJECT_ID and not GEMINI_API_KEY:
    choice = input("¿Deseas usar (1) API Key de Google AI Studio o (2) Proyecto de Google Cloud Vertex AI? [1/2]: ").strip()
    if choice == "2":
        PROJECT_ID = input("Ingresa tu PROJECT_ID de Google Cloud: ").strip()
    else:
        GEMINI_API_KEY = input("Ingresa tu GEMINI_API_KEY: ").strip()

if PROJECT_ID:
    LOCATION = "global"
    client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)
    print(f"✓ Conectado a Colab Enterprise: project={PROJECT_ID}, location={LOCATION}")
elif GEMINI_API_KEY:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("✓ Conectado mediante Gemini API Key")
else:
    raise ValueError("Debes especificar un PROJECT_ID de Google Cloud o una GEMINI_API_KEY.")

# =========================================================================
# 🎯 MODELOS OFICIALES DE GENMEDIA
# =========================================================================
TEXT_MODEL = "gemini-3.7-flash"
IMAGE_MODEL = "gemini-3.1-flash-image"       # Nano Banana 2
TTS_MODEL = "gemini-3.1-flash-tts-preview"   # Voice Director TTS
MUSIC_MODEL = "lyria-3-clip-preview"         # Lyria 3
VIDEO_MODEL = "gemini-omni-flash-preview"     # Gemini Omni Flash

# Directorio de entregables
OUT = Path("media/taller_genmedia")
OUT.mkdir(parents=True, exist_ok=True)
print(f"✓ Directorio de entregables listo: {OUT.resolve()}")


In [ ]:
def _bytes(data):
    return data if isinstance(data, bytes) else base64.b64decode(data)

def _get_item_from_interaction(interaction, kind):
    direct = getattr(interaction, f"output_{kind}", None)
    if direct is not None:
        return direct
    for step in getattr(interaction, "steps", None) or []:
        if getattr(step, "type", None) != "model_output":
            continue
        for item in getattr(step, "content", None) or []:
            if getattr(item, "type", None) == kind:
                return item
    return None

def image_part(path):
    path = Path(path)
    return {
        "type": "image",
        "data": base64.b64encode(path.read_bytes()).decode("utf-8"),
        "mime_type": mimetypes.guess_type(path.name)[0] or "image/png",
    }

def video_part(path):
    path = Path(path)
    return {
        "type": "video",
        "data": base64.b64encode(path.read_bytes()).decode("utf-8"),
        "mime_type": mimetypes.guess_type(path.name)[0] or "video/mp4",
    }

def save_image(interaction, path):
    item = _get_item_from_interaction(interaction, "image")
    if item is None or not getattr(item, "data", None):
        raise RuntimeError("La interacción no devolvió datos de imagen.")
    path = Path(path)
    path.write_bytes(_bytes(item.data))
    display(DisplayImage(filename=str(path), width=720))
    print(f"✓ Imagen guardada en: {path}")
    return path

def save_audio(interaction, path, is_pcm=False):
    item = _get_item_from_interaction(interaction, "audio")
    if item is None or not getattr(item, "data", None):
        raise RuntimeError("La interacción no devolvió datos de audio.")
    raw = _bytes(item.data)
    path = Path(path)
    if is_pcm and not raw.startswith(b"RIFF"):
        with wave.open(str(path), "wb") as wav:
            wav.setnchannels(1)
            wav.setsampwidth(2)
            wav.setframerate(24000)
            wav.writeframes(raw)
    else:
        path.write_bytes(raw)
    display(Audio(filename=str(path)))
    print(f"✓ Audio guardado en: {path}")
    return path

def _download_gcs_or_http(uri, path):
    if uri.startswith("gs://"):
        subprocess.run(["gcloud", "storage", "cp", uri, str(path)], check=True)
        return
    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    credentials.refresh(GoogleAuthRequest())
    sep = "&" if "?" in uri else "?"
    req = urllib.request.Request(
        f"{uri}{sep}alt=media",
        headers={"Authorization": f"Bearer {credentials.token}"},
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        path.write_bytes(resp.read())

def save_video(interaction, path):
    item = _get_item_from_interaction(interaction, "video")
    if item is None:
        raise RuntimeError("La interacción no devolvió video.")
    path = Path(path)
    if getattr(item, "data", None):
        path.write_bytes(_bytes(item.data))
    elif getattr(item, "uri", None):
        _download_gcs_or_http(item.uri, path)
    else:
        raise RuntimeError("El video no contiene datos binarios ni URI descargable.")
    
    video_b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    html_video = f'''
    <div style="margin:12px 0;">
      <video width="680" height="382" controls style="border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,0.15);">
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        Tu navegador no soporta el elemento de video.
      </video>
    </div>
    '''
    display(HTML(html_video))
    print(f"✓ Video guardado en: {path}")
    return path


## 1. Brief Creativo Maestro con Gemini 3.7 Flash
Definimos un esquema estrictamente tipado con **Pydantic** para que Gemini genere el brief JSON que orquestará todas las modalidades.


In [ ]:
class CreativeBrief(BaseModel):
    titulo: str = Field(description="Título cinematográfico de la campaña")
    logline: str = Field(description="Sinopsis narrativa en 2 oraciones")
    escena: str = Field(description="Descripción detallada de la escena y personaje central")
    direccion_visual_5_capas: str = Field(description="Prompt en inglés en 5 capas (Subject, Action, Composition 16:9, Lighting, Art Finish) para Nano Banana")
    voice_director_prompt: str = Field(description="Prompt completo con estructura Voice Director: Profile, Scene, Director Notes, Transcript con tags")
    locucion_texto: str = Field(description="Texto exacto de la frase locutada en español")
    musica_prompt: str = Field(description="Prompt musical en 4 pilares (Genre, Instruments, 72 BPM, Negative constraints) para Lyria")
    omni_video_timeline: str = Field(description="Timeline descriptiva [0-2s], [2-4s], [4-5s] y diseño de audio para Gemini Omni")

brief_prompt = """
Diseña la microcampaña cinematográfica para una biblioteca nocturna y mágica de Bogotá iluminada por letras doradas flotantes.
En cinco segundos, una lectora abre un libro antiguo y las letras se convierten en luciérnagas doradas que iluminan las estanterías de madera noble.
Tono: Íntimo, mágico, místico y moderno.

Asegúrate de:
1. 'direccion_visual_5_capas' en inglés con subject, action, composition (16:9 35mm lens), lighting (warm amber vs teal rim light), finish.
2. 'voice_director_prompt' usando el framework Voice Director: # AUDIO PROFILE, ## THE SCENE, ### DIRECTOR'S NOTES, #### TRANSCRIPT con tags [serenity], [awe], [whispers], [short pause].
3. 'musica_prompt' en inglés para Lyria con 72 BPM, instrumentación sutil y restricciones negativas.
4. 'omni_video_timeline' con marcas [0-2s], [2-4s], [4-5s] y sección de Audio.
"""

brief_ix = client.interactions.create(
    model=TEXT_MODEL,
    input=brief_prompt,
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": CreativeBrief,
        "temperature": 0.7,
    }
)

brief = json.loads(brief_ix.output_text)
(OUT / "brief.json").write_text(json.dumps(brief, ensure_ascii=False, indent=2), encoding="utf-8")
print("✓ Brief JSON Maestro generado:")
print(json.dumps(brief, ensure_ascii=False, indent=2))


## 2. Keyframe 16:9 con Nano Banana 2
Aplicamos la técnica de **5 capas de prompting**: Sujeto ➔ Acción/Pose ➔ Composición 16:9 ➔ Iluminación ➔ Acabado/Render.


In [ ]:
image_prompt = f"""
Master cinematic opening keyframe in 16:9 widescreen format.

SUBJECT & ACTION:
{brief['escena']}

COMPOSITION & LENS:
16:9 aspect ratio, eye-level 35mm lens, f/1.8 shallow depth of field. One central protagonist seated at an antique polished mahogany table opening a weathered leather-bound tome. Glowing golden script fireflies rise toward towering shadowed library shelves.

LIGHTING & COLOR:
Dramatic volumetric amber lamp glow contrasted against deep nocturnal teal shadows. Subsurface scattering on skin, luminescent gold particles catching dust motes.

ART FINISH & TEXTURE:
{brief['direccion_visual_5_capas']}
Photorealistic, 8k octane render, hyper-detailed, clean render without text, logos or extra duplicated limbs.
""".strip()

print("Generando Keyframe 16:9 con Nano Banana 2...")
image_ix = client.interactions.create(
    model=IMAGE_MODEL,
    input=image_prompt,
    generation_config={
        "aspect_ratio": "16:9"
    }
)

keyframe_path = save_image(image_ix, OUT / "01_keyframe.png")


## 3. Locución Profesional con Voice Director TTS
Utilizamos el framework **Voice Director** para dirigir al modelo `gemini-3.1-flash-tts-preview` como a un actor de doblaje profesional con notas de interpretación y **Inline Audio Tags** (`[serenity]`, `[awe]`, `[whispers]`, `[short pause]`).


In [ ]:
print("Generando Locución Profesional con Voice Director Framework...")
print("=" * 65)
print(brief['voice_director_prompt'])
print("=" * 65)

tts_ix = client.interactions.create(
    model=TTS_MODEL,
    input=brief['voice_director_prompt']
)

voice_path = save_audio(tts_ix, OUT / "02_locucion.wav", is_pcm=True)


## 4. Banda Sonora Instrumental con Lyria 3
Componemos una pista instrumental de 30 segundos con estructura armónica adaptada a 72 BPM y ambientación mágica.


In [ ]:
music_prompt = brief['musica_prompt']
print(f"Generando Soundtrack con Lyria 3:\n{music_prompt}\n")

music_ix = client.interactions.create(
    model=MUSIC_MODEL,
    input=music_prompt,
    generation_config={
        "duration_seconds": 30
    }
)

music_path = save_audio(music_ix, OUT / "03_musica.mp3")


## 5. Video Cinemático & Reemplazo con Gemini Omni Flash
Exploramos los 3 modos de generación de video con `gemini-omni-flash-preview`:
1. **Text-to-Video** con Timeline Prompting.
2. **First-Frame to Video** animando el keyframe generado (`<FIRST_FRAME>`).
3. **Video-to-Video Character Swap** transformando el personaje del video a partir de una foto de referencia.


In [ ]:
# =========================================================================
# EJERCICIO 5.1: Text-to-Video con Timeline Prompting
# =========================================================================
text_video_prompt = f"""
Continuous unbroken five-second cinematic shot in 16:9 widescreen, no jump cuts. Slow push-in toward an antique wooden table inside a nocturnal mystical library.

{brief['omni_video_timeline']}

Camera: 35mm eye-level smooth push-in, shallow depth of field, warm amber illumination.
Audio: Soft page rustle, gentle room acoustics, delicate crystalline shimmer. No spoken dialogue.
No watermarks, logos, or sudden scene cuts.
""".strip()

print("Generando Video 5.1 (Text-to-Video) con Gemini Omni Flash...")
text_video_ix = client.interactions.create(
    model=VIDEO_MODEL,
    input=text_video_prompt,
    generation_config={
        "duration_seconds": 5,
        "aspect_ratio": "16:9"
    }
)

text_video_path = save_video(text_video_ix, OUT / "04_omni_texto_video.mp4")


In [ ]:
# =========================================================================
# EJERCICIO 5.2: First-Frame to Video (<FIRST_FRAME>)
# =========================================================================
image_video_prompt = f"""
<FIRST_FRAME> Use the supplied image as the exact opening keyframe.
Continuous unbroken five-second shot, no scene cuts. Preserve the exact reader, facial features, wardrobe, lighting, and library environment.

[0-2s] The reader gently lifts the leather cover of the antique book; soft amber light breathes outward from the pages.
[2-4s] Luminous glowing letter fireflies rise in a gentle spiral, casting dancing gold caustics across her face.
[4-5s] She smiles warmly, her eyes following a rising light as the camera settles in 16:9.

Camera: Very subtle organic push-in.
Audio: Delicate page turning sound, distant night breeze, magical glass chime resonance. No dialogue.
No captions, logos, or new unwanted characters.
""".strip()

print("Generando Video 5.2 (First-Frame to Video) con Gemini Omni Flash...")
image_video_ix = client.interactions.create(
    model=VIDEO_MODEL,
    input=[
        image_part(keyframe_path),
        {"type": "text", "text": image_video_prompt}
    ],
    generation_config={
        "duration_seconds": 5,
        "aspect_ratio": "16:9"
    }
)

image_video_path = save_video(image_video_ix, OUT / "05_omni_keyframe_video.mp4")


In [ ]:
# =========================================================================
# EJERCICIO 5.3: Reemplazo de Personaje en Video con Imagen de Referencia
# =========================================================================
character_swap_prompt = """
[# Sources <VIDEO_REF_0>] [# References <IMAGE_REF_0>]
Replace the reader in the video with the character from <IMAGE_REF_0>.
Preserve the exact hand motion of opening the book, body posture, head rotation, timing, and framing from <VIDEO_REF_0>, while transforming the character entirely into a futuristic cosmic explorer (translucent crystalline textures, subtle luminescent cybernetic accents, and starlight reflections). Maintain photorealistic temporal consistency across all frames. Keep background library environment consistent.
""".strip()

print("Generando Video 5.3 (Video-to-Video Character Swap) con Gemini Omni Flash...")
character_swap_ix = client.interactions.create(
    model=VIDEO_MODEL,
    input=[
        video_part(image_video_path),
        image_part(keyframe_path),
        {"type": "text", "text": character_swap_prompt}
    ],
    generation_config={
        "duration_seconds": 5,
        "aspect_ratio": "16:9"
    }
)

character_swap_path = save_video(character_swap_ix, OUT / "06_omni_character_swap.mp4")


In [ ]:
expected_assets = {
    "Brief JSON": OUT / "brief.json",
    "Keyframe 16:9": OUT / "01_keyframe.png",
    "Locución Voice Director": OUT / "02_locucion.wav",
    "Soundtrack Lyria 3": OUT / "03_musica.mp3",
    "Video 1 (Text-to-Video)": OUT / "04_omni_texto_video.mp4",
    "Video 2 (First-Frame)": OUT / "05_omni_keyframe_video.mp4",
    "Video 3 (Character Swap)": OUT / "06_omni_character_swap.mp4",
}

print("=" * 70)
print("📦 INVENTARIO DE ENTREGABLES MULTIMEDIA")
print("=" * 70)
for label, path in expected_assets.items():
    exists = path.exists() and path.stat().st_size > 0
    status = "✅ OK" if exists else "❌ FALTA"
    size = f"{path.stat().st_size / 1024:.1f} KB" if exists else "—"
    print(f"{status:8} | {label:24} | {size:>10} | {path.name}")
print("=" * 70)
